[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rkarmaka/multi-agent-course/blob/main/modules/Module_5_Multi_Agents/agent_subagent_orchestrator_starter.ipynb)

# Agent → Sub-Agents → Orchestrator

Three levels of building with the **Anthropic SDK**, from simplest to most powerful. Each section is one idea:

| Level | Pattern | Core idea |
|-------|---------|-----------|
| 1 | **Agent** | One LLM call with a *skill* (a saved system prompt). |
| 2 | **Sub-Agents** | Specialists run in their own context window; the main agent sees only their *results*. |
| 3 | **Orchestrator** | A lead plans, specialists work in parallel, a tech lead integrates. |

Run the cells top to bottom.

## Setup

Install the SDK and set your API key. The client reads `ANTHROPIC_API_KEY` from the environment.

In [1]:
%pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 14.3 MB/s eta 0:00:00


In [3]:
import os

# If it isn't already in your environment, uncomment and set it here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# 1. Google Colab Secrets (the recommended path on Colab).
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

# 2. Local fallback: .env file or an exported environment variable.
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    api_key = os.environ.get("ANTHROPIC_API_KEY")

MODEL = "claude-opus-4-8"  # current Opus; swap to claude-sonnet-4-6 for cheaper/faster runs

---
## 1. Agent

An **agent** at its simplest is one LLM call with a *skill* loaded as the system prompt.

A **skill is just a `.md` file** — a saved, reusable prompt. Here we write one to disk, then load it as the `system` prompt.

In [11]:
import os
import urllib.request
from pathlib import Path

from IPython.display import Markdown, display

def show(text):
    """Render a string as rich markdown in the notebook output."""
    display(Markdown(text))

os.makedirs("skills", exist_ok=True)

# Raw GitHub URL of the skill that ships with market-sizing-sdk.
SKILL_NAME = "market-sizing"
RAW_URL = f"https://github.com/hamzafarooq/multi-agent-course/blob/main/.claude/skills/{SKILL_NAME}/SKILL.md"
skill_path = Path("skill/SKILL.md")

if not skill_path.exists():
    skill_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW_URL, skill_path)
    print(f"Downloaded skill -> {skill_path}")
else:
    print(f"Skill already present -> {skill_path}")

with open(f"skills/{SKILL_NAME}.md", "w") as f:
    f.write(
        "You are a full-stack web developer. Given a project brief, produce a "
        "complete plan: tech stack, file structure, and key code."
    )

print(f"Wrote skills/{SKILL_NAME}.md")

Skill already present -> skill/SKILL.md
Wrote skills/market-sizing.md


In [12]:
import anthropic

client = anthropic.Anthropic(api_key=api_key)

# Load the .md file as the system prompt
with open("skills/full-stack-developer.md") as f:
    skill_prompt = f.read()

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=skill_prompt,
    messages=[{
        "role": "user",
        "content": "Do market sizing for SpaceX.",
    }],
)

result = response.content[0].text
show(result)

# Market Sizing for SpaceX

Quick note: you asked a full-stack developer for market sizing — so I'll give you a proper business analysis here. (If you actually wanted a web app to *visualize* market sizing, say the word and I'll spec out the build.)

---

## Framework: Segment-by-Segment TAM/SAM/SOM

SpaceX operates across distinct markets with very different dynamics. I'll size each, then aggregate.

### 1. Launch Services
| Metric | Value | Logic |
|--------|-------|-------|
| **TAM** | ~$15–18B/yr | Global orbital launch + national security + commercial payloads |
| **SAM** | ~$10B/yr | Markets open to a US commercial provider (excludes China, sovereign-mandated launches) |
| **SOM** | ~$6–7B/yr | SpaceX already commands ~60–65% of global commercial launch by revenue |

**Drivers:** Falcon 9 reusability dropped cost/kg to ~$2,000–3,000. Starship targets sub-$500/kg, which *expands the TAM itself* by enabling new payload classes.

### 2. Satellite Internet (Starlink) — the real value driver
| Metric | Value | Logic |
|--------|-------|-------|
| **TAM** | ~$1T+ | Global broadband/connectivity spend |
| **SAM** | ~$50–80B/yr | Underserved/unserved + mobility (maritime, aviation, RVs) + enterprise/gov |
| **SOM** | ~$10–15B/yr by 2027 | ~5M+ subscribers × ~$1,000–1,500 ARPU + enterprise contracts |

**Bottom-up Starlink check:**
```
Addressable households (rural/underserved, OECD + emerging):  ~300M
Realistic penetration at price point:                          3–5%
Subscribers:                                                   ~9–15M
Blended ARPU (consumer + premium + mobility):                  ~$1,200/yr
Revenue range:                                                 ~$11B–18B/yr
```
Plus **Direct-to-Cell** (partnership with T-Mobile et al.) — a potential multi-billion adjacency.

### 3. Government / Defense (NASA, DoD, Space Force)
- NASA Commercial Crew + Cargo: ~$2–3B in active contract value
- Starshield (military Starlink variant): multi-billion classified pipeline
- HLS (Human Landing System / Artemis): ~$4B contract value
- **Estimated annual run-rate:** ~$3–4B

### 4. Emerging / Optionality
- Point-to-point Earth transport, lunar/Mars cargo, in-space manufacturing, space tourism (Polaris). Highly speculative — model as **option value**, not base-case revenue.

---

## Aggregate View

| Market | TAM | SpaceX SOM (near-term) |
|--------|-----|------------------------|
| Launch | ~$15B | ~$6–7

---
## 2. Sub-Agents

A **sub-agent** runs in its *own isolated context window*. The main agent never sees the sub-agent's reasoning — only the final result it returns. This keeps the main agent's context clean and focused.

Below: a **researcher** sub-agent and a **formatter** sub-agent each do their own work, then the **main planner** combines just their two outputs.

In [14]:
# Sub-agent: research runs in its own isolated context window.
# The planner never sees its reasoning, only the result.
research_resp = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system="You are a tech researcher.",
    messages=[{
        "role": "user",
        "content": "Best stack for a task manager app? Keep it brief.",
    }],
)
stack_research = research_resp.content[0].text

# Sub-agent: a formatter that turns any plan into a clean sprint-card structure.
format_resp = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system="You are a sprint formatter. Convert any plan into a clean sprint card structure.",
    messages=[{
        "role": "user",
        "content": "Format spec: 2 sprints, 5 days each, with owner and status fields.",
    }],
)
format_spec = format_resp.content[0].text

# Main agent: sees only the two results — not the reasoning behind them.
plan_resp = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are a project planner.",
    messages=[{
        "role": "user",
        "content": (
            f"Build a task manager sprint plan.\n"
            f"Stack research:\n{stack_research}\n\n"
            f"Format spec:\n{format_spec}"
        ),
    }],
)

show(plan_resp.content[0].text)

# Sprint Plan

---

## Sprint 1
**Goal:** Project setup, database schema, and authentication

**Duration:** 5 days

| Day | Task | Owner | Status |
|-----|------|-------|--------|
| 1 | Initialize Next.js + Tailwind project, configure repo & CI | Frontend Dev | ☐ Not Started |
| 2 | Provision Supabase (Postgres + Auth), connect Prisma ORM | Backend Dev | ☐ Not Started |
| 3 | Design DB schema (users, tasks, due dates, reminders) | Backend Dev | ☐ Not Started |
| 4 | Run Prisma migrations, seed test data | Backend Dev | ☐ Not Started |
| 5 | Implement Auth.js sign-up/login flows + session handling | Full-stack Dev | ☐ Not Started |

**Sprint 1 Owner:** _[lead]_
**Sprint 1 Status:** ☐ Not Started

---

## Sprint 2
**Goal:** Core task CRUD functionality

**Duration:** 5 days

| Day | Task | Owner | Status |
|-----|------|-------|--------|
| 1 | Build task API routes (create, read, update, delete) | Backend Dev | ☐ Not Started |
| 2 | Create task list UI with Tailwind components | Frontend Dev | ☐ Not Started |
| 3 | Implement task creation/edit forms with validation | Frontend Dev | ☐ Not Started |
| 4 | Add due date picker and task completion toggle | Frontend Dev | ☐ Not Started |
| 5 | Wire UI to API, handle loading/error states | Full-stack Dev | ☐ Not Started |

**Sprint 2 Owner:** _[lead]_
**Sprint 2 Status:** ☐ Not Started

---

## Sprint 3
**Goal:** Reminders, multi-user features, and filtering

**Duration:** 5 days

| Day | Task | Owner | Status |
|-----|------|-------|--------|
| 1 | Implement reminder scheduling logic & notifications | Backend Dev | ☐ Not Started |
| 2 | Add task filtering/sorting (by date, status, priority) | Frontend Dev | ☐ Not Started |
| 3 | Build multi-user task ownership & permissions | Backend Dev | ☐ Not Started |
| 4 | Add realtime updates via Supabase subscriptions | Full-stack Dev | ☐ Not Started |
| 5 | Implement user dashboard with task summary | Frontend Dev | ☐ Not Started |

**Sprint 3 Owner:** _[lead]_
**Sprint 3 Status:** ☐ Not Started

---

## Sprint 4
**Goal:** Testing, polish, and deployment

**Duration:** 5 days

| Day | Task | Owner | Status |
|-----|------|-------|--------|
| 1 | Write unit

---
## 3. Orchestrator

The full pattern: a **project lead** scopes the work, a **team of specialists** works **in parallel**, and a **tech lead** integrates everything into one plan.

We use the **async client** so the four specialists run concurrently instead of one after another.

In [17]:
import asyncio
import anthropic

aclient = anthropic.AsyncAnthropic(api_key=api_key)


async def run(system: str, prompt: str) -> str:
    """One async LLM call with a role (system) and a task (prompt)."""
    r = await aclient.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text


async def orchestrate(goal: str) -> str:
    # Step 1: Project lead scopes the work into specialist briefs.
    brief = await run(
        "You are a project lead. Break the goal into specialist briefs "
        "for your team. Return JSON.",
        goal,
    )

    # Step 2: All specialists work in parallel.
    design, frontend, backend, qa = await asyncio.gather(
        run("You are a UI Designer.",        f"Brief: {brief}"),
        run("You are a Frontend Developer.", f"Brief: {brief}"),
        run("You are a Backend Developer.",  f"Brief: {brief}"),
        run("You are a QA Engineer.",        f"Brief: {brief}"),
    )

    # Step 3: Tech lead integrates everything into one build plan.
    final = await run(
        "You are a Tech Lead.",
        f"Integrate into one build plan:\n"
        f"Design: {design}\nFrontend: {frontend}\n"
        f"Backend: {backend}\nQA: {qa}",
    )
    return final


goal = "Build a full-stack task manager with React, FastAPI, and PostgreSQL."

# In a notebook, await the coroutine directly (Jupyter runs an event loop for you).
final_plan = await orchestrate(goal)
show(final_plan)

# Unified Build Plan — Full-Stack Task Manager

This plan integrates the Database, Backend, Frontend, Design, and QA deliverables into a single sequenced execution path with clear contracts between layers.

---

## 1. Foundational Contracts (resolve before coding)

These decisions block everyone downstream. Lock them first.

| Decision | Resolution | Affects |
|----------|-----------|---------|
| Enum values | `status`: `todo`/`in_progress`/`done` · `priority`: `low`/`med`/`high` | DB, BE, FE types, Design color map |
| Enum implementation | PG `ENUM` type (QA finding #5) | DB, BE |
| `task_tags` PK | Composite `PRIMARY KEY (task_id, tag_id)` (QA #1) | DB |
| `tags` uniqueness | `UNIQUE (user_id, name)` (QA #2) | DB, BE validation |
| Email handling | `CITEXT` + lowercase normalization (QA #6) | DB, BE auth |
| FK delete rules | See §2 | DB, BE, FE empty states |
| `updated_at` | DB trigger (not app-layer) (QA #4) | DB, BE |

**Action:** Lock this table as the source of truth. All four contributors sign off before Phase 1.

---

## 2. Delete Behavior Matrix (cross-cutting)

| Relationship | Rule | Rationale |
|--------------|------|-----------|
| `user` → `projects

---
## Recap

- **Agent** — one call + a skill (saved system prompt).
- **Sub-Agents** — specialists in isolated context; the main agent sees only results.
- **Orchestrator** — plan → parallel specialists → integrate.

Same building block (`messages.create`) every time — only the *coordination* changes.